<a href="https://colab.research.google.com/github/jumafernandez/modelado-gestion-datos-untref/blob/main/codigo/r_postgresql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Conectar R con PostgreSQL

**Modelado y Gestión de Datos - UNTREF - 2026**

En este notebook nos conectamos desde R a un servidor PostgreSQL del curso, le enviamos consultas SQL y recibimos los resultados como *data frames*. El servidor tiene dos bases de solo lectura:

- `w3schools`, la base de práctica del TP0302.
- `eph`, con los microdatos de la EPH que armamos en el cuaderno [De la EPH a PostgreSQL](https://jumafernandez.github.io/modelado-gestion-datos-untref/guias/cuaderno-eph/).

Cada celda se ejecuta con **Shift + Enter** o con el botón ▶ de su izquierda.

## 1. El paquete

RPostgres conecta R con PostgreSQL. Colab no lo trae instalado, así que se instala al empezar cada sesión.

In [ ]:
install.packages("RPostgres")
library(RPostgres)

## 2. Conectarse a la base

`dbConnect()` abre la conexión con los mismos datos que usamos en pgAdmin: el servidor, el puerto, la base, el usuario y la clave. Antes de ejecutar la celda hay que reemplazar `CLAVE` por la clave del usuario `alumno`, que está en el campus.

In [ ]:
con <- dbConnect(Postgres(),
                 host     = "170.210.103.79",
                 port     = 80,
                 dbname   = "w3schools",
                 user     = "alumno",
                 password = "CLAVE")

dbListTables(con)   # las tablas de la base

## 3. Una consulta, un data frame

`dbGetQuery()` envía una consulta SQL y devuelve el resultado como data frame. Es la misma consulta que escribiríamos en pgAdmin; esta responde la primera pregunta del Anexo I del TP0302. PostgreSQL guarda los nombres de tablas y columnas en minúscula, y así aparecen en el data frame.

In [ ]:
productos <- dbGetQuery(con, "
  SELECT productname, price
  FROM products
  WHERE price > 50
  ORDER BY price DESC")

productos

## 4. Una consulta agrupada y un gráfico

Conviene que PostgreSQL resuelva las juntas y los agrupamientos, y que a R llegue solo el resultado: acá, una fila por categoría con el total vendido.

In [ ]:
ventas <- dbGetQuery(con, "
  SELECT c.categoryname AS categoria,
         SUM(od.quantity * p.price) AS ventas
  FROM orderdetails od
  INNER JOIN products p   ON od.productid = p.productid
  INNER JOIN categories c ON p.categoryid = c.categoryid
  GROUP BY c.categoryname
  ORDER BY ventas DESC")

ventas

In [ ]:
library(ggplot2)

ggplot(ventas, aes(x = ventas, y = reorder(categoria, ventas))) +
  geom_col(fill = "#E08A1E") +
  labs(title = "Ventas por categoría", x = "Total vendido", y = NULL) +
  theme_minimal()

## 5. Cerrar la conexión

Al terminar, se cierra la conexión con el servidor.

In [ ]:
dbDisconnect(con)

---

# La EPH desde R

Ahora, el mismo camino con datos reales: la base `eph`, con los microdatos de la Encuesta Permanente de Hogares de INDEC desde 2018. El análisis usa dplyr y ggplot2, que Colab ya trae instalados.

In [ ]:
library(dplyr)

con <- dbConnect(Postgres(),
                 host     = "170.210.103.79",
                 port     = 80,
                 dbname   = "eph",
                 user     = "alumno",
                 password = "CLAVE")

dbListTables(con)

## Las personas de un trimestre

De una tabla grande conviene pedir solo lo que se va a usar: `personas` tiene más de un millón y medio de filas. La consulta une cada persona con su hogar (para el ponderador) y llega hasta la región. Trae el primer trimestre de 2026; para analizar otro, se cambian los valores del `WHERE`.

In [ ]:
consulta_personas <- "
SELECT p.sexo,
       p.edad,
       p.id_condicion_actividad,
       r.nombre AS region,
       h.pondera
FROM personas p
INNER JOIN hogares h     ON p.codusu = h.codusu AND p.anio = h.anio
                        AND p.trimestre = h.trimestre AND p.nro_hogar = h.nro_hogar
INNER JOIN viviendas v   ON h.codusu = v.codusu
INNER JOIN aglomerados a ON v.id_aglomerado = a.id
INNER JOIN regiones r    ON a.id_region = r.id
WHERE p.anio = 2026 AND p.trimestre = 1"

personas <- dbGetQuery(con, consulta_personas)

glimpse(personas)
nrow(personas)            # personas encuestadas
sum(personas$pondera)     # población que representan

## Tasas del mercado de trabajo por región

Condición de actividad: 1 ocupado, 2 desocupado, 3 inactivo, 4 menor de 10 años. Las tasas suman el ponderador, como las de INDEC:

- actividad = (ocupados + desocupados) / población
- empleo = ocupados / población
- desocupación = desocupados / (ocupados + desocupados)

In [ ]:
tasas_region <- personas |>
  group_by(region) |>
  summarise(
    poblacion   = sum(pondera),
    ocupados    = sum(pondera[id_condicion_actividad == 1]),
    desocupados = sum(pondera[id_condicion_actividad == 2])
  ) |>
  mutate(
    tasa_actividad    = round(100 * (ocupados + desocupados) / poblacion, 1),
    tasa_empleo       = round(100 * ocupados / poblacion, 1),
    tasa_desocupacion = round(100 * desocupados / (ocupados + desocupados), 1)
  ) |>
  select(region, tasa_actividad, tasa_empleo, tasa_desocupacion) |>
  arrange(desc(tasa_desocupacion))

tasas_region

## La serie de la desocupación

Para recorrer todos los trimestres no hace falta traer el millón y medio de personas: PostgreSQL agrupa y calcula la tasa, y llega una fila por trimestre.

In [ ]:
consulta_serie <- "
SELECT p.anio,
       p.trimestre,
       ROUND(100.0 * SUM(CASE WHEN p.id_condicion_actividad = 2 THEN h.pondera ELSE 0 END)
                   / SUM(CASE WHEN p.id_condicion_actividad IN (1, 2) THEN h.pondera ELSE 0 END),
             1) AS tasa_desocupacion
FROM personas p
INNER JOIN hogares h ON p.codusu = h.codusu AND p.anio = h.anio
                    AND p.trimestre = h.trimestre AND p.nro_hogar = h.nro_hogar
GROUP BY p.anio, p.trimestre
ORDER BY p.anio, p.trimestre"

serie <- dbGetQuery(con, consulta_serie)
serie

In [ ]:
# Para el eje del gráfico: 2026.00 es el primer trimestre, 2026.25 el segundo...
serie <- serie |>
  mutate(periodo = anio + (trimestre - 1) / 4)

ggplot(serie, aes(x = periodo, y = tasa_desocupacion)) +
  geom_line(colour = "#E08A1E", linewidth = 1) +
  geom_point(colour = "#1C1A3A", size = 1.5) +
  scale_x_continuous(breaks = 2018:2026) +
  labs(title = "Tasa de desocupación por trimestre",
       x = NULL, y = "% de la población económicamente activa",
       caption = "Fuente: EPH (INDEC), 31 aglomerados urbanos.") +
  theme_minimal()

## Estructura por edad y sexo

Con el data frame de las personas del trimestre, agrupado en tramos de cinco años.

In [ ]:
etiquetas_edad <- c(paste(seq(0, 75, by = 5), seq(4, 79, by = 5), sep = "-"), "80 y más")

piramide <- personas |>
  mutate(
    grupo_edad = cut(edad, breaks = c(seq(0, 80, by = 5), Inf),
                     right = FALSE, labels = etiquetas_edad),
    sexo = if_else(sexo == 1, "Varones", "Mujeres")
  ) |>
  group_by(grupo_edad, sexo) |>
  summarise(poblacion = sum(pondera), .groups = "drop") |>
  mutate(porcentaje = 100 * poblacion / sum(poblacion),
         porcentaje = if_else(sexo == "Varones", -porcentaje, porcentaje))

ggplot(piramide, aes(x = porcentaje, y = grupo_edad, fill = sexo)) +
  geom_col(width = 0.9) +
  scale_x_continuous(labels = abs) +
  scale_fill_manual(values = c(Mujeres = "#E08A1E", Varones = "#1C1A3A")) +
  labs(title = "Estructura por edad y sexo, primer trimestre de 2026",
       x = "% de la población", y = NULL, fill = NULL,
       caption = "Fuente: EPH (INDEC), 31 aglomerados urbanos.") +
  theme_minimal()

## Ingreso per cápita familiar por región

La vista `ingresos_hogares` del cuaderno ya lo calcula. Para los ingresos, INDEC usa el ponderador PONDIH; los hogares que no respondieron tienen PONDIH = 0 y quedan afuera. La mediana ponderada es el primer valor en el que el peso acumulado llega a la mitad.

In [ ]:
consulta_ingresos <- "
SELECT r.nombre AS region,
       i.ingreso_per_capita,
       i.pondih
FROM ingresos_hogares i
INNER JOIN viviendas v   ON i.codusu = v.codusu
INNER JOIN aglomerados a ON v.id_aglomerado = a.id
INNER JOIN regiones r    ON a.id_region = r.id
WHERE i.anio = 2026 AND i.trimestre = 1 AND i.pondih > 0"

ingresos <- dbGetQuery(con, consulta_ingresos)

mediana_ponderada <- function(x, w) {
  orden <- order(x)
  peso_acumulado <- cumsum(w[orden])
  x[orden][which(peso_acumulado >= sum(w) / 2)[1]]
}

ingresos |>
  group_by(region) |>
  summarise(
    hogares_encuestados = n(),
    media   = round(weighted.mean(ingreso_per_capita, pondih)),
    mediana = round(mediana_ponderada(ingreso_per_capita, pondih))
  ) |>
  arrange(desc(mediana))

In [ ]:
# La distribución es muy asimétrica: el eje va en escala logarítmica.
# Los hogares con ingreso cero no tienen logaritmo y no se grafican.
ingresos |>
  filter(ingreso_per_capita > 0) |>
  ggplot(aes(x = ingreso_per_capita, weight = pondih)) +
  geom_density(fill = "#F3C98B", colour = "#A3540A") +
  scale_x_log10(labels = scales::label_number(big.mark = ".", decimal.mark = ",")) +
  facet_wrap(vars(region), ncol = 2) +
  labs(title = "Ingreso per cápita familiar, primer trimestre de 2026",
       x = "Pesos por mes (escala logarítmica)", y = NULL,
       caption = "Fuente: EPH (INDEC), ponderado con PONDIH.") +
  theme_minimal() +
  theme(axis.text.y = element_blank())

## Cerrar la conexión

In [ ]:
dbDisconnect(con)

---

## La contraseña, fuera del código

En este notebook la clave se escribe en el código porque el usuario `alumno` es de solo lectura y su clave es pública para el curso. Con credenciales propias no conviene: un script con la contraseña escrita termina compartido por mail o subido a un repositorio. En RStudio, la práctica habitual es guardarla en el archivo `.Renviron` de tu usuario, que R lee al iniciar:

1. Abrir el archivo con `file.edit("~/.Renviron")`.
2. Agregar la línea `PGPASSWORD=tu_contraseña`, con un Enter al final, y guardar.
3. Reiniciar R (en RStudio, *Session > Restart R*).

Desde ese momento, la conexión se escribe sin la contraseña:

```r
con <- dbConnect(Postgres(),
                 host     = "localhost",
                 port     = 5432,
                 dbname   = "eph",
                 user     = "postgres",
                 password = Sys.getenv("PGPASSWORD"))
```